# Monte Carlo와 Q-learning 비교 실험

이 노트북은 **ipynb 단독 제출용**입니다. 외부 `environment.py`, `compare_training.py`, `MonteCarlo/mc_agent.py`, `QLearning/q_agent.py` 파일을 import하지 않고, 필요한 전체 코드를 이 노트북 안에 포함했습니다.

## 교재 기반 부분

- Monte Carlo: 에피소드 종료 후 return을 계산하여 Q-table 갱신
- Q-learning: 매 스텝마다 TD target을 이용하여 Q-table 갱신
- 정책 선택: epsilon-greedy 방식

## 교재와 다르게 한 부분

- Monte Carlo와 Q-learning을 같은 Cliff Walking 환경에서 비교하기 위해 공통 환경 클래스를 작성했습니다.
- 교재 4장의 신경망은 사용하지 않고, 교재 2장의 표 기반 방식에 맞춰 Q-table을 사용했습니다.
- 보고서 분석을 위해 평균 보상, 평균 이동 횟수, 절벽 추락 횟수, 성공률, 학습된 경로 이미지를 추가했습니다.


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm


MAX_STEPS = 200
EPISODES = 20000
EVAL_EPISODES = 50
OUTPUT_DIR = "팀플"


## 1. 환경 구현

4 x 12 Cliff Walking 환경입니다. 시작점은 왼쪽 아래, 목표점은 오른쪽 아래입니다. 절벽에 빠지면 -100 보상을 받고 시작점으로 돌아가며, 목표점에 도착하면 +100 보상을 받고 에피소드가 종료됩니다.

In [ ]:
class Environment:
    def __init__(self):
        self.road = -1
        self.cliff = -100
        self.goal = 100

        self.start_position = np.array([3, 0])
        self.goal_position = np.array([3, 11])
        self.action = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])

        self.reward = np.full((4, 12), self.road)
        self.reward[3, 1:11] = self.cliff
        self.reward[3, 11] = self.goal

        self.reward_list1 = np.full((4, 12), "road", dtype=object)
        self.reward_list1[3, 1:11] = "cliff"
        self.reward_list1[3, 11] = "goal"

    def move(self, agent, action):
        pos = np.array(agent.get_pos())
        next_pos = pos + self.action[action]

        if not self._is_inside(next_pos):
            next_pos = pos

        reward = self.reward[next_pos[0], next_pos[1]]
        done = reward == self.goal

        if reward == self.cliff:
            next_pos = self.start_position.copy()

        agent.set_pos(next_pos)
        return next_pos, reward, done

    def _is_inside(self, pos):
        return 0 <= pos[0] < self.reward.shape[0] and 0 <= pos[1] < self.reward.shape[1]


## 2. Monte Carlo 구현

교재 2장의 Monte Carlo control 방식을 사용했습니다. 에피소드 동안 `(state, action, reward)`를 저장하고, 에피소드가 끝난 뒤 뒤에서부터 return `G`를 계산하여 Q-table을 갱신합니다.

교재와 다르게 한 부분은 Cliff Walking 좌표 상태에 맞춰 `Q[row, col, action]` 형태의 3차원 Q-table을 사용하고, 비교 실험을 위해 에이전트를 클래스로 구성한 점입니다.

In [ ]:
class MCAgent:
    """Monte Carlo control agent using a Q-table."""

    def __init__(
        self,
        state_shape=(4, 12),
        action_size=4,
        gamma=0.99,
        epsilon=1.0,
        epsilon_decay=0.9995,
        epsilon_min=0.05,
        first_visit=True,
    ):
        self.state_shape = state_shape
        self.action_size = action_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.first_visit = first_visit

        self.pos = np.array([3, 0])
        self.action = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
        self.q_table = np.zeros((*self.state_shape, self.action_size))
        self.q_visit = np.zeros((*self.state_shape, self.action_size))
        self.memory = []

    def set_pos(self, position):
        self.pos = np.array(position)
        return self.pos

    def get_pos(self):
        return self.pos

    def select_action(self, state=None):
        pos = self.pos if state is None else np.array(state)

        if np.random.rand() <= self.epsilon:
            return np.random.randint(self.action_size)

        q_values = self.q_table[pos[0], pos[1], :]
        max_actions = np.flatnonzero(q_values == np.max(q_values))
        return np.random.choice(max_actions)

    def append_sample(self, state, action, reward):
        self.memory.append((np.array(state), action, reward))

    def train_model(self):
        G = 0
        visited = set()

        for state, action, reward in reversed(self.memory):
            G = reward + self.gamma * G
            key = (int(state[0]), int(state[1]), int(action))

            if self.first_visit and key in visited:
                continue

            visited.add(key)
            row, col, act = key
            self.q_visit[row, col, act] += 1
            self.q_table[row, col, act] += (
                (G - self.q_table[row, col, act]) / self.q_visit[row, col, act]
            )

        self.memory = []
        self.decay_epsilon()

    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


## 3. Q-learning 구현

교재 2장의 Q-learning 업데이트를 사용했습니다. Q-learning은 Monte Carlo와 달리 에피소드가 끝날 때까지 기다리지 않고, 매 스텝마다 다음 상태의 최대 Q값을 이용해 현재 Q값을 갱신합니다.

`Q(s,a) <- Q(s,a) + alpha * [r + gamma * max Q(s',a') - Q(s,a)]`

In [ ]:
class QLearningAgent:
    """Q-learning agent using a Q-table."""

    def __init__(
        self,
        state_shape=(4, 12),
        action_size=4,
        learning_rate=0.1,
        gamma=0.99,
        epsilon=1.0,
        epsilon_decay=0.995,
        epsilon_min=0.01,
    ):
        self.state_shape = state_shape
        self.action_size = action_size
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

        self.pos = np.array([3, 0])
        self.action = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
        self.q_table = np.zeros((*self.state_shape, self.action_size))

    def set_pos(self, position):
        self.pos = np.array(position)
        return self.pos

    def get_pos(self):
        return self.pos

    def select_action(self, state=None):
        pos = self.pos if state is None else np.array(state)

        if np.random.rand() <= self.epsilon:
            return np.random.randint(self.action_size)

        q_values = self.q_table[pos[0], pos[1], :]
        max_actions = np.flatnonzero(q_values == np.max(q_values))
        return np.random.choice(max_actions)

    def train_model(self, state, action, reward, next_state, done):
        row, col = int(state[0]), int(state[1])
        next_row, next_col = int(next_state[0]), int(next_state[1])

        now_q = self.q_table[row, col, action]
        next_q = 0 if done else np.max(self.q_table[next_row, next_col, :])
        target = reward + self.gamma * next_q

        self.q_table[row, col, action] += self.learning_rate * (target - now_q)

    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


## 4. 학습 및 평가 함수

이 부분은 두 알고리즘을 같은 조건에서 비교하기 위해 추가한 코드입니다. 학습 중 reward, step, cliff fall, success를 기록하고, 학습 후에는 epsilon을 0으로 두어 greedy 정책 성능을 평가합니다.

In [ ]:
def get_exploring_start_states(env):
    states = []
    for row in range(env.reward.shape[0]):
        for col in range(env.reward.shape[1]):
            if env.reward_list1[row][col] == "road":
                states.append(np.array([row, col]))
    return states


def run_episode(env, agent, algo, train=True, max_steps=MAX_STEPS, start_state=None, first_action=None):
    state = np.array(env.start_position if start_state is None else start_state)
    agent.set_pos(state)

    total_reward = 0
    steps = 0
    falls = 0
    done = False

    while not done and steps < max_steps:
        if steps == 0 and first_action is not None:
            action = first_action
        else:
            action = agent.select_action(state)

        next_state, reward, done = env.move(agent, action)
        next_state = np.array(next_state)

        if train:
            if algo == "MC":
                agent.append_sample(state, action, reward)
            elif algo == "QL":
                agent.train_model(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward
        steps += 1
        falls += int(reward == env.cliff)

    return total_reward, steps, falls, done


def evaluate_agent(agent, algo, episodes=EVAL_EPISODES):
    env = Environment()
    old_epsilon = agent.epsilon
    agent.epsilon = 0.0

    rewards = []
    success_steps = []
    falls = 0
    successes = 0

    for _ in range(episodes):
        reward, steps, episode_falls, success = run_episode(env, agent, algo, train=False)
        rewards.append(reward)
        falls += episode_falls

        if success:
            successes += 1
            success_steps.append(steps)

    agent.epsilon = old_epsilon

    return {
        "EvalReward": np.mean(rewards),
        "EvalSteps": np.mean(success_steps) if success_steps else np.nan,
        "EvalFalls": falls,
        "EvalSuccessRate": successes / episodes,
    }


def train_agent(algo, episodes=EPISODES, seed=0):
    np.random.seed(seed)
    env = Environment()
    agent = MCAgent() if algo == "MC" else QLearningAgent()
    exploring_states = get_exploring_start_states(env)

    rewards = []
    steps = []
    falls = []
    successes = []

    for _ in tqdm(range(episodes), desc=algo):
        if algo == "MC":
            start_state = exploring_states[np.random.randint(len(exploring_states))]
            first_action = np.random.randint(agent.action_size)
        else:
            start_state = None
            first_action = None

        reward, episode_steps, episode_falls, success = run_episode(
            env,
            agent,
            algo,
            train=True,
            start_state=start_state,
            first_action=first_action,
        )

        if algo == "MC":
            agent.train_model()
        else:
            agent.decay_epsilon()

        rewards.append(reward)
        steps.append(episode_steps)
        falls.append(episode_falls)
        successes.append(int(success))

    return {
        "agent": agent,
        "rewards": np.array(rewards),
        "steps": np.array(steps),
        "falls": np.array(falls),
        "successes": np.array(successes),
        "eval": evaluate_agent(agent, algo),
    }


## 5. 결과 출력 및 시각화 함수

학습 곡선과 학습된 greedy 경로를 이미지로 출력합니다. 이 시각화 코드는 교재 코드에 없지만, 보고서의 비교 분석을 위해 추가했습니다.

In [ ]:
def summarize_result(algo, result, last_n=50):
    return {
        "Algo": algo,
        "TrainRewardLastN": np.mean(result["rewards"][-last_n:]),
        "TrainStepsLastN": np.mean(result["steps"][-last_n:]),
        "TrainFalls": int(np.sum(result["falls"])),
        "TrainSuccessRate": np.mean(result["successes"]),
        **result["eval"],
    }


def print_summary(rows):
    print("\n" + "=" * 118)
    print(" " * 42 + "MC vs Q-Learning Comparison")
    print("=" * 118)
    print(
        f"{'Algo':<12} | {'Train Reward':<13} | {'Train Steps':<12} | {'Train Falls':<11} | "
        f"{'Train Success':<13} | {'Eval Reward':<11} | {'Eval Steps':<10} | {'Eval Success':<12}"
    )
    print("-" * 118)

    for row in rows:
        eval_steps = "N/A" if np.isnan(row["EvalSteps"]) else f"{row['EvalSteps']:.2f}"
        print(
            f"{row['Algo']:<12} | {row['TrainRewardLastN']:<13.2f} | {row['TrainStepsLastN']:<12.2f} | "
            f"{row['TrainFalls']:<11} | {row['TrainSuccessRate'] * 100:<12.1f}% | "
            f"{row['EvalReward']:<11.2f} | {eval_steps:<10} | {row['EvalSuccessRate'] * 100:<11.1f}%"
        )

    print("=" * 118)


def plot_training(mc_result, ql_result):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    plt.figure(figsize=(14, 8))

    plt.subplot(2, 2, 1)
    plt.plot(mc_result["rewards"], label="Monte Carlo")
    plt.plot(ql_result["rewards"], label="Q-Learning")
    plt.title("Training Reward")
    plt.xlabel("Episode")
    plt.ylabel("Total Reward")
    plt.legend()

    plt.subplot(2, 2, 2)
    plt.plot(mc_result["steps"], label="Monte Carlo")
    plt.plot(ql_result["steps"], label="Q-Learning")
    plt.title("Training Steps")
    plt.xlabel("Episode")
    plt.ylabel("Steps")
    plt.legend()

    plt.subplot(2, 2, 3)
    plt.plot(np.cumsum(mc_result["successes"]) / (np.arange(len(mc_result["successes"])) + 1), label="Monte Carlo")
    plt.plot(np.cumsum(ql_result["successes"]) / (np.arange(len(ql_result["successes"])) + 1), label="Q-Learning")
    plt.title("Cumulative Success Rate")
    plt.xlabel("Episode")
    plt.ylabel("Success Rate")
    plt.legend()

    plt.subplot(2, 2, 4)
    plt.plot(np.cumsum(mc_result["falls"]), label="Monte Carlo")
    plt.plot(np.cumsum(ql_result["falls"]), label="Q-Learning")
    plt.title("Cumulative Cliff Falls")
    plt.xlabel("Episode")
    plt.ylabel("Falls")
    plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "learning_comparison_mc_vs_ql.png"))


def get_greedy_path(agent, max_steps=MAX_STEPS):
    env = Environment()
    old_epsilon = agent.epsilon
    agent.epsilon = 0.0

    state = np.array(env.start_position)
    agent.set_pos(state)

    path = [tuple(state)]
    rewards = []
    actions = []
    done = False

    for _ in range(max_steps):
        action = agent.select_action(state)
        next_state, reward, done = env.move(agent, action)
        next_state = np.array(next_state)

        actions.append(action)
        rewards.append(reward)
        path.append(tuple(next_state))

        state = next_state
        if done:
            break

    agent.epsilon = old_epsilon
    return path, actions, rewards, done


def plot_policy_path(agent, title, filename):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    env = Environment()
    path, actions, rewards, success = get_greedy_path(agent)

    rows, cols = env.reward.shape
    color_grid = np.zeros((rows, cols))

    for row in range(rows):
        for col in range(cols):
            if env.reward_list1[row][col] == "cliff":
                color_grid[row, col] = -1
            elif env.reward_list1[row][col] == "goal":
                color_grid[row, col] = 2

    for row, col in path:
        if env.reward_list1[row][col] == "road":
            color_grid[row, col] = 1

    start = tuple(env.start_position)
    goal = tuple(env.goal_position)
    color_grid[start] = 3
    color_grid[goal] = 2

    cmap = plt.matplotlib.colors.ListedColormap([
        "#ef4444",
        "#f8fafc",
        "#60a5fa",
        "#22c55e",
        "#facc15",
    ])
    bounds = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
    norm = plt.matplotlib.colors.BoundaryNorm(bounds, cmap.N)

    plt.figure(figsize=(14, 5))
    plt.imshow(color_grid, cmap=cmap, norm=norm)
    plt.xticks(range(cols))
    plt.yticks(range(rows))
    plt.grid(which="major", color="#334155", linewidth=1)
    plt.tick_params(bottom=False, left=False)

    for idx, (row, col) in enumerate(path):
        label = "S" if (row, col) == start else "G" if (row, col) == goal else str(idx)
        plt.text(col, row, label, ha="center", va="center", color="#0f172a", fontsize=10, fontweight="bold")

    action_symbol = {0: "U", 1: "R", 2: "D", 3: "L"}
    for (row, col), action in zip(path[:-1], actions):
        plt.text(col + 0.28, row - 0.28, action_symbol[action], ha="center", va="center", color="#111827", fontsize=12)

    total_reward = sum(rewards)
    plt.title(f"{title} Greedy Path | Success: {success} | Steps: {len(actions)} | Reward: {total_reward}")
    plt.tight_layout()
    plt.savefig(filename)
    return path, actions, rewards, success


def plot_all_policy_paths(mc_result, ql_result):
    mc_path = plot_policy_path(
        mc_result["agent"],
        "Monte Carlo",
        os.path.join(OUTPUT_DIR, "monte_carlo_greedy_path.png"),
    )
    ql_path = plot_policy_path(
        ql_result["agent"],
        "Q-Learning",
        os.path.join(OUTPUT_DIR, "q_learning_greedy_path.png"),
    )
    return mc_path, ql_path


## 6. 실험 실행

In [ ]:
mc_result = train_agent("MC", episodes=EPISODES, seed=0)
ql_result = train_agent("QL", episodes=EPISODES, seed=0)


In [ ]:
rows = [
    summarize_result("Monte Carlo", mc_result),
    summarize_result("Q-Learning", ql_result),
]

print_summary(rows)


In [ ]:
plot_training(mc_result, ql_result)
plt.show()


In [ ]:
path_results = plot_all_policy_paths(mc_result, ql_result)
plt.show()

path_results


## 7. 비교 분석

Monte Carlo는 실제 에피소드에서 얻은 return을 기반으로 Q값을 갱신하기 때문에 전체 경로의 결과를 직접 반영합니다. 하지만 에피소드가 끝난 뒤에만 학습할 수 있고, return의 변동이 커서 안정적인 정책을 얻기까지 시간이 걸릴 수 있습니다.

Q-learning은 매 스텝마다 TD target을 이용해 즉시 Q값을 갱신합니다. 따라서 절벽에 빠지는 경험과 목표에 가까워지는 경험을 빠르게 반영할 수 있으며, 이 환경에서는 상대적으로 짧고 안정적인 경로를 학습하는 경향을 보입니다.

본 구현은 교재 2장의 표 기반 강화학습 알고리즘을 중심으로 작성했으며, 교재 4장의 클래스 구조와 결과 비교 시각화 방식을 참고했습니다. 교재 4장의 신경망은 사용하지 않았고, Q-table 기반으로 Monte Carlo와 Q-learning을 비교했습니다.